# Polymorphism

> Generated from [`04-Polymorphism.py`](04-Polymorphism.py) by `tools/build_notebooks.py` - edit the `.py` (or its `.md` notes), not this notebook. Changes made here are overwritten on the next build.

## Quick notes

### What this covers
"Many forms" — the same method name or operator behaving differently depending on the object it's used with. Covers method overriding, duck typing, callable objects, and comparison-related dunder methods.

### Key points
- **Polymorphism** = the same operation (method call, operator) behaves differently depending on the actual type of the object — `len()` and `+` already do this for built-in types.
- **Via inheritance/overriding**: a shared method name (`speak()`) defined differently in each subclass — calling `animal.speak()` in a loop runs the right version for whatever type `animal` actually is.
- **Duck typing**: Python doesn't require a common base class — any object with the right method/attribute works ("if it walks like a duck and quacks like a duck..."). A function can accept anything with a `.speak()` method, no shared `Animal` ancestor needed.
- **`__call__`**: defining it lets instances be invoked like functions (`obj(value)`) — the same call syntax works for functions, lambdas, and objects alike.
- **`functools.singledispatch`**: one function with different implementations chosen by the *type* of its first argument — polymorphism resolved by argument type rather than by which object's method runs.
- Operator overloading (`__add__`, `__eq__`, etc., detailed in 06-Dataclasses-And-Operator-Overloading.py) is polymorphism too — it lets `+`/`==`/`<` "do the right thing" for custom objects.
- ABCs (see 05-Abstraction.py) can force subclasses to implement a shared method, guaranteeing polymorphism works consistently across all of them.

### Gotchas (the part worth revising twice)
- **`__eq__` and `__hash__` are linked**: by default, objects are hashable (by `id()`) and compared by identity. The moment you override `__eq__` to compare by value, Python automatically sets `__hash__` to `None` — the object becomes unhashable (can't go in a `set` or be a dict key) unless you also define `__hash__` explicitly, consistent with your `__eq__`.
- **`functools.total_ordering`**: writing all of `__lt__`, `__le__`, `__gt__`, `__ge__` by hand is repetitive — define `__eq__` plus just ONE of the others, and `@total_ordering` fills in the rest.
- Duck typing means there's no compile-time or import-time check that an object has the right method — mismatches only surface as an `AttributeError` at call time.
- `singledispatch` dispatches on the type of the *first argument only*, and falls back to the generic (undecorated) implementation for any type without a registered overload.

### When it matters
Reach for polymorphism whenever you want one function/loop to work uniformly across several types now, and to keep working when new types (e.g. a new `Bird` class) are added later, without touching the existing code.

### One-line recap example
```python
animals = [Dog(), Cat()]
for animal in animals:
    print(animal.speak())   # same call, different behavior per actual type
```

## 01. Polymorphism - Introduction

- "Poly" (many) + "morph" (forms) - the same operation/method name behaves
  differently depending on the object it's called on.
- Lets you write code that works with objects of different types uniformly.

## 02. Built-in Polymorphism (recap)

- len() and + already behave differently depending on the type.

In [ ]:
print("len('hello'):", len("hello"))
print("len([1, 2, 3]):", len([1, 2, 3]))
print("'a' + 'b':", "a" + "b")
print("[1] + [2]:", [1] + [2])

## 03. Polymorphism via Method Overriding (inheritance)

In [ ]:
class Animal:
    def speak(self):
        return "Some generic animal sound"

class Dog(Animal):
    def speak(self):
        return "Woof!"

class Cat(Animal):
    def speak(self):
        return "Meow!"

print("\n# 03. Polymorphism via Method Overriding")
animals = [Animal(), Dog(), Cat()]
for animal in animals:
    print(f"{animal.__class__.__name__}: {animal.speak()}")
# - Same method call `animal.speak()`, different behavior per actual type.

## 04. Duck Typing - "If it walks like a duck..."

- Python doesn't require a common base class for polymorphism.
- Any object with the expected method/attribute works - type isn't checked.

In [ ]:
class Duck:
    def speak(self):
        return "Quack!"

class Robot:
    def speak(self):
        return "Beep boop, I am speaking"

print("\n# 04. Duck Typing")
for thing in [Duck(), Robot()]:
    print(thing.speak())  # works for both - neither inherits from Animal

## 05. Polymorphism with Functions

In [ ]:
def make_it_speak(entity):
    # works with ANY object that has a .speak() method
    print(entity.speak())

print("\n# 05. Polymorphism with Functions")
make_it_speak(Dog())
make_it_speak(Duck())

## 06. Operator Overloading is Polymorphism Too (recap)

- See 06-Dataclasses-And-Operator-Overloading.py for __add__, __eq__, etc.

## 07. Polymorphism with Abstract Base Classes (preview)

- Covered fully in 05-Abstraction.py - ABCs force subclasses to implement
  a shared method, guaranteeing polymorphism works consistently.

## 08. Why Polymorphism Matters

## 09. __eq__ and __hash__ Relationship

- By default, objects are hashable using their id() and compared with ==
  by identity too - that's how they end up in sets/dict keys "for free".
- Overriding __eq__ to compare by VALUE makes Python set __hash__ to None
  automatically - the object becomes unhashable, since equal-by-value
  objects would need equal hashes, and the default id()-based hash can't
  guarantee that anymore.

In [ ]:
class PointNoHash:
    def __init__(self, x, y):
        self.x, self.y = x, y
    def __eq__(self, other):
        return self.x == other.x and self.y == other.y
    # no __hash__ defined -> Python sets it to None automatically

print("\n# 09. __eq__ and __hash__ Relationship")
p1 = PointNoHash(1, 2)
print("p1 == PointNoHash(1, 2):", p1 == PointNoHash(1, 2))
try:
    {p1} # attempting to put it in a set requires __hash__
except TypeError as e:
    print("Caught (unhashable):", e)

class PointHashable(PointNoHash):
    def __hash__(self):
        return hash((self.x, self.y)) # must be consistent with __eq__

print("With __hash__ defined, {PointHashable(1, 2)} works:", {PointHashable(1, 2)})

## 10. Full Comparison Dunders + functools.total_ordering

- __lt__, __le__, __gt__, __ge__ each back one comparison operator.
- Writing all four (plus __eq__) is repetitive - @functools.total_ordering
  fills in the rest as long as you define __eq__ and just ONE of the others.

In [ ]:
from functools import total_ordering

@total_ordering
class Money:
    def __init__(self, amount):
        self.amount = amount
    def __eq__(self, other):
        return self.amount == other.amount
    def __lt__(self, other):
        return self.amount < other.amount
    # __le__, __gt__, __ge__ are derived automatically from __eq__ and __lt__

print("\n# 10. functools.total_ordering")
m1, m2 = Money(100), Money(200)
print("m1 < m2:", m1 < m2)
print("m1 <= m2 (derived):", m1 <= m2)
print("m1 > m2 (derived):", m1 > m2)
print("m1 >= m2 (derived):", m1 >= m2)

## 11. Polymorphism via __call__ (Callable Instances)

- Defining __call__ lets instances be invoked like functions: obj(...).
- It's polymorphism because the SAME syntax (calling with parentheses)
  works uniformly whether the callee is a function, a lambda, or an object.

In [ ]:
class Multiplier:
    def __init__(self, factor):
        self.factor = factor
    def __call__(self, value):
        return value * self.factor

print("\n# 11. Polymorphism via __call__")
double = Multiplier(2)
triple = Multiplier(3)
print("double(5):", double(5))   # called just like a function
print("triple(5):", triple(5))
print("callable(double):", callable(double))

## 12. Single Dispatch Generic Functions (functools.singledispatch)

- singledispatch lets one function have different implementations chosen
  by the TYPE of its first argument - polymorphism resolved by argument
  type rather than by which object's method is called.

In [ ]:
from functools import singledispatch

@singledispatch
def describe(value):
    return f"a value: {value}"

@describe.register
def _(value: int):
    return f"an integer: {value}"

@describe.register
def _(value: list):
    return f"a list with {len(value)} items"

print("\n# 12. functools.singledispatch")
print("describe(3.14):", describe(3.14))   # falls back to the generic implementation
print("describe(5):", describe(5))         # dispatches to the int implementation
print("describe([1, 2, 3]):", describe([1, 2, 3])) # dispatches to the list implementation